#### Notebook for running React experiments

In [2]:
import sys, os
sys.path.append('..')
root  = '../root/'

In [3]:
import joblib
from util import summarize_react_trial, log_react_trial, save_agents
from agents import ReactReflectAgent, ReactAgent, ReflexionStrategy

#### Load the HotpotQA Sample

In [7]:
hotpot = joblib.load('../data/hotpot-qa-distractor-sample.joblib').reset_index(drop = True)

print(type(hotpot))
hotpot.columns
hotpot.iloc[0]

#Xd, ok, so they only tested on hard questions
hard_ones = hotpot[hotpot["level"] == "hard"]
hard_ones

<class 'pandas.core.frame.DataFrame'>


,id,question,answer,type,level,supporting_facts,context
0,5a7613c15542994ccc9186bf,VIVA Media AG changed it's name in 2004. What ...,Gesellschaft mit beschränkter Haftung,bridge,hard,"{'title': ['VIVA Media', 'Gesellschaft mit bes...","{'title': ['Constantin Medien', 'VIVA Poland',..."
1,5adf2fa35542993344016c11,Which of Jonny Craig and Pete Doherty has been...,"Jonny"" Craig",comparison,hard,"{'title': ['Jonny Craig', 'Jonny Craig', 'Pete...","{'title': ['Pete Doherty', 'Relativity (Emaros..."
2,5adfdef9554299025d62a36b,Where was the first governor after the The Mis...,"Bath, Maine",bridge,hard,"{'title': ['Maine gubernatorial election, 1820...","{'title': ['Compromise of 1790', 'Anti-Nebrask..."
3,5a7180205542994082a3e856,"The creator of ""Wallace and Gromit"" also creat...",Creature Comforts,bridge,hard,"{'title': ['Creature Comforts', 'Creature Comf...","{'title': ['Creature Comforts', 'Tata Steel Zo..."
4,5a78bc6b554299148911f979,Woman's Era and Naj are what kind of magazines?,fortnightly women interest magazine,comparison,hard,"{'title': ['Woman's Era', 'Naj'], 'sent_id': [...","{'title': ['Lifestyle trends and media', 'Chin..."
...,...,...,...,...,...,...,...
95,5ac509e95542994611c8b333,The chicken is a type of dance pattern that is...,the Twist,bridge,hard,"{'title': ['Chicken (dance)', 'Chicken (dance)...","{'title': ['Dance move', 'Decoded neurofeedbac..."
96,5a8b42be55429949d91db515,What profession does John Lanchester and Alan ...,novelist,comparison,hard,"{'title': ['John Lanchester', 'Alan Dean Foste...","{'title': ['Bloodhype', 'Alan Dean Foster', 'T..."
97,5ab9b2c7554299743d22ebaf,Are both Lygodium or Maxillaria a genus of orc...,no,comparison,hard,"{'title': ['Lygodium', 'Maxillaria'], 'sent_id...","{'title': ['Paracaleana', 'Heterotaxis', 'Orni..."
98,5a7b63eb55429931da12ca7e,What city does Paul Clyne and David Soares hav...,New York,bridge,hard,"{'title': ['Paul Clyne', 'David Soares'], 'sen...","{'title': ['Rhabdodontidae', 'Paul Clyne', 'Tu..."


#### Define the Reflexion Strategy

In [4]:
print(ReflexionStrategy.__doc__)


    NONE: No reflection
    LAST_ATTEMPT: Use last reasoning trace in context 
    REFLEXION: Apply reflexion to the next reasoning trace 
    LAST_ATTEMPT_AND_REFLEXION: Use last reasoning trace in context and apply reflexion to the next reasoning trace 
    


In [12]:
strategy: ReflexionStrategy = ReflexionStrategy.REFLEXION

In [ ]:
#TODO: Sanity check on few basic examples to verify that everything works the way its supposed to, and if it does, create a seperate 
#agent/llm class just for the reflector, and pass that into each react agent

In [ ]:

#Basically, the way this currently works inside my head, 

#n trials loop -> we don't want it structured this way at all. We want it to run the n trials question after question
#after that, run the



#### Initialize a React Agent for each question

In [13]:
agent_cls = ReactReflectAgent if strategy != ReflexionStrategy.NONE else ReactAgent
agents = [agent_cls(row['question'], row['answer']) for _, row in hotpot.iterrows()]

#### Run `n` trials

In [14]:
n = 5
trial = 0
log = ''

In [ ]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        if strategy != ReflexionStrategy.NONE:
            agent.run(reflect_strategy = strategy)
        else:
            agent.run()
        print(f'Answer: {agent.key}')
    trial += 1
    log += log_react_trial(agents, trial)
    correct, incorrect, halted = summarize_react_trial(agents)
    print(f'Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}, Halted: {len(halted)}')

#### Save the result log

In [ ]:
with open(os.path.join(root, 'ReAct', strategy.value, f'{len(agents)}_questions_{trial}_trials.txt'), 'w') as f:
    f.write(log)
save_agents(agents, os.path.join('ReAct', strategy.value, 'agents'))